In [ ]:
# Cell 1 — Config & load
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd
import time
import os
import ssl
import urllib.request

INPUT_CSV  = "processed/person.csv"
OUTPUT_DIR = "processed"
BATCH_SIZE = 50
SLEEP_SEC  = 1.0

# All relationship properties to fetch
# (wikidata_property, output_csv, col_prefix, fetch_qualifiers)
# fetch_qualifiers=True only for spouse (P26) to capture marriage start/end/cause
RELATION_CONFIGS = [
    # ("P3373", "person_siblings.csv",     "sibling",    False),
    # ("P26",   "person_spouses.csv",      "spouse",     True ),
    # ("P40",   "person_children.csv",     "child",      False),
    # ("P1038", "person_relatives.csv",    "relative",   False),
    # ("P22",   "person_fathers.csv",      "father",     False),
    # ("P25",   "person_mothers.csv",      "mother",     False),
    # ("P3448", "person_stepparents.csv",  "stepparent", False),
    # ("P7026", "person_stepchildren.csv", "stepchild",  False),
    # ("P43",   "person_stepfathers.csv",  "stepfather", False),
    # ("P44",   "person_stepmothers.csv",  "stepmother", False),
    # ("P1290", "person_godparents.csv",   "godparent",  False),
    # ("P9150", "person_godchildren.csv",  "godchild",   False),
]

df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df)} rows from {INPUT_CSV}")
df.head()

In [ ]:
# Cell 2 — Helper functions
ssl_context = ssl.create_default_context()
ssl_context.check_hostname = False
ssl_context.verify_mode = ssl.CERT_NONE


def build_values_clause(uris):
    entities = " ".join(f"<{uri}>" for uri in uris)
    return f"VALUES ?person {{ {entities} }}"


def fetch_relation_batch(uris, sparql_endpoint, property_id, fetch_qualifiers=False):
    """Fetch one relationship property for a batch of person URIs.
    fetch_qualifiers=True pulls P580/P582/P1534 marriage qualifiers (spouse only).
    """
    values_clause = build_values_clause(uris)

    if fetch_qualifiers:
        query = f"""
        SELECT ?person ?related ?relatedLabel
               (MIN(?start) AS ?marriageStart)
               (MAX(?end)   AS ?marriageEnd)
               (SAMPLE(?endCause) AS ?endCauseId)
        WHERE {{
          {values_clause}
          OPTIONAL {{
            ?person p:{property_id} ?stmt .
            ?stmt ps:{property_id} ?related .
            ?related rdfs:label ?relatedLabel .
            FILTER(LANG(?relatedLabel) = \"en\")
            OPTIONAL {{ ?stmt pq:P580  ?start    . }}
            OPTIONAL {{ ?stmt pq:P582  ?end      . }}
            OPTIONAL {{ ?stmt pq:P1534 ?endCause . }}
          }}
        }}
        GROUP BY ?person ?related ?relatedLabel
        """
    else:
        query = f"""
        SELECT ?person ?related ?relatedLabel WHERE {{
          {values_clause}
          OPTIONAL {{
            ?person wdt:{property_id} ?related .
            ?related rdfs:label ?relatedLabel .
            FILTER(LANG(?relatedLabel) = \"en\")
          }}
        }}
        """

    sparql_endpoint.setQuery(query)
    sparql_endpoint.setReturnFormat(JSON)

    opener = urllib.request.build_opener(
        urllib.request.HTTPSHandler(context=ssl_context)
    )
    urllib.request.install_opener(opener)
    results = sparql_endpoint.query().convert()

    relation_map = {}  # { person_uri: [ {related_id, related_name, ...}, ... ] }
    for row in results["results"]["bindings"]:
        person_uri   = row["person"]["value"]
        related_uri  = row["related"]["value"]      if "related"      in row else None
        related_name = row["relatedLabel"]["value"] if "relatedLabel" in row else None

        if not related_uri:
            continue

        entry = {"related_id": related_uri, "related_name": related_name}

        if fetch_qualifiers:
            entry["marriage_start"] = row["marriageStart"]["value"] if "marriageStart" in row else None
            entry["marriage_end"]   = row["marriageEnd"]["value"]   if "marriageEnd"   in row else None
            entry["end_cause_id"]   = row["endCauseId"]["value"]    if "endCauseId"    in row else None

        relation_map.setdefault(person_uri, [])
        if entry not in relation_map[person_uri]:
            relation_map[person_uri].append(entry)

    return relation_map


def build_dataframe(relation_map, col_prefix, name_map, fetch_qualifiers=False):
    """Flatten relation_map into a tidy DataFrame."""
    records = []
    for person_uri, related_list in relation_map.items():
        person_qid = person_uri.rsplit("/", 1)[-1]
        for r in related_list:
            related_qid = r["related_id"].rsplit("/", 1)[-1]
            rec = {
                "person_uri"         : person_uri,
                "person_qid"         : person_qid,
                "person_name"        : name_map.get(person_uri),
                f"{col_prefix}_uri"  : r["related_id"],
                f"{col_prefix}_qid"  : related_qid,
                f"{col_prefix}_name" : r["related_name"],
            }
            if fetch_qualifiers:
                end_cause = r.get("end_cause_id")
                rec["marriage_start"] = r.get("marriage_start")
                rec["marriage_end"]   = r.get("marriage_end")
                rec["end_cause_qid"]  = end_cause.rsplit("/", 1)[-1] if end_cause else None
            records.append(rec)
    return pd.DataFrame(records)


print("Helper functions defined ✅")

In [ ]:
# Cell 3 — Run all relationship types
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.addCustomHttpHeader("User-Agent", "RelationFetcher/1.0 (research project)")

all_uris      = df["person"].dropna().unique().tolist()
total_batches = (len(all_uris) + BATCH_SIZE - 1) // BATCH_SIZE
name_map      = df.set_index("person")["name"].to_dict()

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Unique person URIs to query : {len(all_uris)}")
print(f"Relationship types to fetch : {len(RELATION_CONFIGS)}")
print("=" * 65)

summary = {}  # prop -> (persons_with_relation, total_links)

for (prop, output_csv, col_prefix, fetch_qualifiers) in RELATION_CONFIGS:
    print(f"\n▶  [{prop}]  {col_prefix.upper()}  →  {output_csv}")
    print("-" * 55)

    relation_map = {}

    for i in range(0, len(all_uris), BATCH_SIZE):
        batch     = all_uris[i : i + BATCH_SIZE]
        batch_num = i // BATCH_SIZE + 1
        print(f"  Batch {batch_num}/{total_batches} ({len(batch)} entities)...", end=" ")

        try:
            result      = fetch_relation_batch(batch, sparql, prop, fetch_qualifiers)
            relation_map.update(result)
            total_links = sum(len(v) for v in result.values())
            print(f"{len(result)} persons ({total_links} links found)")
        except Exception as e:
            print(f"ERROR: {e}")

        time.sleep(SLEEP_SEC)

    out_df   = build_dataframe(relation_map, col_prefix, name_map, fetch_qualifiers)
    out_path = os.path.join(OUTPUT_DIR, output_csv)
    out_df.to_csv(out_path, index=False)

    summary[prop] = (len(relation_map), len(out_df))
    print(f"  ✅ Saved {out_path}  ({len(out_df)} rows, {len(relation_map)} persons)")

print("\n" + "=" * 65)
print("ALL DONE")

In [ ]:
# Cell 4 — Summary report
print(f"{'Prop':<8} {'Type':<15} {'Output CSV':<32} {'Persons':>8} {'Links':>8}")
print("-" * 75)
for (prop, output_csv, col_prefix, _) in RELATION_CONFIGS:
    persons, links = summary.get(prop, (0, 0))
    print(f"{prop:<8} {col_prefix:<15} {output_csv:<32} {persons:>8,} {links:>8,}")

In [ ]:
# Cell 5 — Optional: inspect any output CSV
sample_file = "processed/person_spouses.csv"   # change to any output file
sample_df   = pd.read_csv(sample_file)
print(f"{sample_file}: {len(sample_df)} rows")
sample_df.head(10)

In [ ]:
# Cell 6 — Optional: retry a specific property for persons not yet in its CSV
# Useful if some batches errored mid-run.
# Set RETRY_PROP to the Wikidata property you want to retry.

RETRY_PROP = "P26"   # change as needed

config = next((c for c in RELATION_CONFIGS if c[0] == RETRY_PROP), None)
if config is None:
    print(f"Property {RETRY_PROP} not found in RELATION_CONFIGS")
else:
    prop, output_csv, col_prefix, fetch_qualifiers = config
    out_path = os.path.join(OUTPUT_DIR, output_csv)

    if os.path.exists(out_path):
        existing_df   = pd.read_csv(out_path)
        queried_qids  = set(existing_df["person_qid"].astype(str).unique())
        retry_uris    = [u for u in all_uris if u.rsplit("/", 1)[-1] not in queried_qids]
    else:
        existing_df  = pd.DataFrame()
        retry_uris   = all_uris

    print(f"Retrying {len(retry_uris)} persons for [{prop}] {col_prefix}...")
    retry_map           = {}
    total_retry_batches = (len(retry_uris) + BATCH_SIZE - 1) // BATCH_SIZE

    for i in range(0, len(retry_uris), BATCH_SIZE):
        batch     = retry_uris[i : i + BATCH_SIZE]
        batch_num = i // BATCH_SIZE + 1
        print(f"  Retry batch {batch_num}/{total_retry_batches}...", end=" ")
        try:
            result = fetch_relation_batch(batch, sparql, prop, fetch_qualifiers)
            retry_map.update(result)
            print(f"{len(result)} found")
        except Exception as e:
            print(f"ERROR: {e}")
        time.sleep(SLEEP_SEC * 2)

    if retry_map:
        retry_df  = build_dataframe(retry_map, col_prefix, name_map, fetch_qualifiers)
        merged_df = pd.concat([existing_df, retry_df], ignore_index=True).drop_duplicates()
        merged_df.to_csv(out_path, index=False)
        print(f"\n✅ Updated {out_path} ({len(merged_df)} rows total)")
    else:
        print("No new results found.")